# 노드 A — 학습 (tonight)

**2026-08-24 밤 · 6 GPU = 3노드 x 2 GPU. 이 노트북은 노드 A 전용.**

세 노드 전부 `exp5_tonight.py` 하나만 부른다. 잡 정의 · 우선순위 · 스킵 · 집계가 전부 거기 있다.

| 노드 | GPU | 역할 | 오늘 밤 산출 |
|---|---|---|---|
| **A** | 0,1 | 학습 2잡 (~15h) | 0셀 → **내일 오후 해금** |
| **B** | 0,1 | eval (우선순위 짝수) | ~12셀 |
| **C** | 0,1 | eval (우선순위 홀수) | ~12셀 |

역할은 고정이 아니다 — 3)번 `suggest()`가 인벤토리를 보고 조정한다
(ready eval 큐가 얕으면 그 GPU를 학습으로 돌린다).

**예산**: eval 1셀(LIBERO-10 x 50ep/task = 500ep) ≈ 2 GPU-h · 학습 1잡(150k) ≈ 15 GPU-h
(해준님 8/11 실측 "500ep 2시간, 5000ep 하루" 기준)

---

## 이 노드가 하는 일

학습 큐 앞에서 2잡을 꺼내 2 GPU로 동시에 돌린다. **셀이 15시간 블로킹**되니 걸어놓고 건드리지 말 것.

기본 순서 1·2번은:

1. **`bimamba_pure`** (K=100) — 지금 `MODEL_CONFIGS['bimamba']`는 `_CARRY_ON` + `use_chunk_pairs=True`로
   학습돼 있어서, eval에서 `sscp_enabled=false`를 걸어도 **학습이 chunk-pair라 순수 BiMamba가 아니다**.
   은지님이 8/18에 "bimamba only를 전부 켜고 돌려서 다시 측정해야겠다"고 하신 그 문제다.
   헤드라인 주장이 "BiMamba only가 긴 chunk에서 ACT를 이긴다"인데 이 ckpt 없이는 리뷰어한테 바로 깨진다.
2. **`act_k150`** — ACT 붕괴 곡선의 오른쪽 끝점. K=50/100 ACT는 이미 있으니 150만 있으면
   K=50→100→150 곡선이 닫힌다. BiMamba+TE K=150 = 26.8이 이미 있어서 act+TE 150만 나오면 구간 완성.

**이미 학습된 건 큐에서 자동으로 빠지고, 중단된 것(PART)은 resume된다.** 인벤토리 결과에 따라
1·2번이 달라질 수 있으니 3)번 셀 출력을 꼭 보고 4)번 dry-run으로 커맨드를 확인할 것.



## 1) 부팅

In [ ]:
import sys
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()          # common_final reload + 태그 등록 (순서 중요)

# 이 노드에서 쓸 GPU. 한 노드에서 창을 2개 띄울 때만 [2, 3] 처럼 직접 지정.
GPUS = v23.available_gpus([0, 1])
print('GPUS =', GPUS)


## 2) 인벤토리 — 뭐가 학습돼 있고 뭐가 없나

서버 파일시스템을 실제로 스캔한다. `MISS` = 학습 필요, `PART` = 중단됨(resume 대상).

In [ ]:
rows = X.inventory()


## 3) 역할 배정

eval은 ckpt가 있어야 돌아간다. **ready eval 큐가 얕으면 그 GPU는 학습에 주는 게 맞다** — 학습만이 내일 이후의 eval을 열어주기 때문이다. `suggest()`가 인벤토리를 보고 정해준다.

| ready eval | 배정 |
|---|---|
| ≥24셀 | A=학습, B·C=eval (둘 다 포화) |
| 1~23셀 | A·C=학습, B=eval |
| 0셀 | A·B·C 전부 학습 |
| 학습 큐 0 | B·C=eval |

In [ ]:
roles = X.suggest()

# 위 출력이 "기본값과 다르다" 라고 하면, 알려주는 한 줄을 **세 노트북 모두**에 붙여넣고
# 이 셀 아래를 다시 실행할 것. (세 노드가 같은 역할표를 봐야 잡이 안 겹친다)
# X.ROLE_OVERRIDE = {'C': 'train'}


## 4) 오늘 밤 계획

전체 학습 큐와 이 노드 몫(2잡)을 같이 찍는다.

In [ ]:
plan = X.plan('A', GPUS)


## 5) dry-run — 커맨드 눈으로 확인 (필수)

**`bimamba_pure` 커맨드에 `--use_chunk_pairs`가 없고 `--policy.sscp_enabled=false`가 있는지** 반드시 확인하고 넘어갈 것. 이거 하나 틀리면 15시간을 날린다.

In [ ]:
X.run_trains(plan['train'], plan['gpus'], dry=True)


## 6) 실행 — 학습 (15시간 블로킹)

여기서부터 커널을 건드리지 말 것. 로그는 `outputs/final/_logs/train__*.log`.


In [ ]:
X.run_trains(plan['train'], plan['gpus'])


## 7) 끝나면 — 다음 잡 이어받기

위 셀이 끝나면 큐가 줄어든다. 이 셀을 실행하면 다음 2잡을 이어서 건다.

In [ ]:
plan = X.plan('A', GPUS)
X.run_trains(plan['train'], plan['gpus'])


## 아침에 볼 것

`X.report()` 가 TE 표 + 레짐 맵을 전부 찍는다. 판단 기준:

| 결과 | 다음 |
|---|---|
| `act+TE@K50` **<** `bimamba+TE@K50` | 크로스오버가 K<50 → ACT K=20/15/10 학습이 급함 (노드 A 내일 큐) |
| `act+TE@K50` **>** `bimamba+TE@K50` | **크로스오버 = K 50~100 확정.** 논문을 "long-chunk regime"으로 리라이트 시작 |
| 레짐 맵에서 `bimamba_cpoff`가 긴 stride에서 `act` 상회 | `bimamba_pure`(A에서 학습중)로 확정 → seed 1,2 추가 |
| `carry`가 짧은 stride에서 `act` 상회 | 레짐 논문 확정 ("두 메커니즘은 보완재가 아니라 대체재") |
| 아무것도 ACT를 못 이김 | TE 축이 유일한 카드 → 거기로 올인 |

**주의**: seed 1개 · 500ep 기준 binomial SE ≈ ±2.2%p. **5%p 미만 차이는 주장하지 말 것.**


In [ ]:
X.report()
